# Assignment 3 - Data exploration and preprocessing

This notebook is deliberately written as a decision log. Each code cell is
preceded by an analysis cell that explains why the next action is being run.

Main target: understand the text collection, identify risks in preprocessing,
and define separate feature strategies for clustering and anomaly detection.


## Run 1 - Analysis / rationale

Before modeling, the input contract has to be verified. The assignment requires
preserving `articles.csv` order when writing `clusters.csv`, and exactly 50
document IDs in `anomalies.csv`. The first action loads all templates and checks
their shape and key columns.


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "articles.csv").exists() and (ROOT.parent / "articles.csv").exists():
    ROOT = ROOT.parent
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)

articles = pd.read_csv(ROOT / "articles.csv")
clusters_template = pd.read_csv(ROOT / "clusters.csv")
anomalies_template = pd.read_csv(ROOT / "anomalies.csv")

print("articles:", articles.shape)
print("clusters template:", clusters_template.shape)
print("anomalies template:", anomalies_template.shape)
print("article columns:", articles.columns.tolist())

assert articles.columns.tolist() == ["doc_id", "text"]
assert clusters_template["doc_id"].tolist() == articles["doc_id"].tolist()
assert anomalies_template.shape[0] == 50
assert articles["doc_id"].is_unique

display(articles.head())


## Run 2 - Analysis / rationale

Text clustering can fail if the model learns length, boilerplate, URLs, e-mail
addresses, or random codes instead of topic vocabulary. I therefore compute both
content-neutral structural features and unsafe-website indicators before any
clustering.


In [ ]:
def clean_text_for_clustering(text: str) -> str:
    """Remove artifacts that are not topical content.

    The goal is not to erase the document meaning. The goal is to stop the model
    from clustering on Usenet signatures, URLs, e-mail addresses, and random IDs.
    """
    text = str(text)

    # Common Usenet signature separator. Apply only if the separator is late
    # enough to plausibly be a signature, not a normal sentence break.
    matches = list(re.finditer(r"\s--\s", text))
    if matches:
        last = matches[-1]
        if last.start() > len(text) * 0.45 or len(text) - last.start() < 900:
            text = text[: last.start()]

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", " ", text)
    text = re.sub(r"\b[A-Z0-9]{5,}(?:-[A-Z0-9]{2,})+\b", " ", text)
    text = re.sub(r"[_=]{2,}|-{3,}", " ", text)
    return text


def build_unsafe_features(texts: pd.Series) -> pd.DataFrame:
    """Features aimed at unsafe/corrupted marketplace-like documents.

    These are intentionally different from topic-clustering features. A document
    can be anomalous because of structure and repeated machine-like tokens, even
    if its individual words are common.
    """
    rows = []
    for text in texts.astype(str):
        urls = len(re.findall(r"https?://\S+|www\.\S+", text))
        listing = len(re.findall(r"LISTING_ID_", text))
        ecommerce = len(
            re.findall(
                r"\b(?:SKU|TOKEN|COUPON|PROMO(?:_ID|_REF)?|BUNDLE_REF|ORDER_REF|SELLER_CODE|PRICE_REF|SALE_CODE)\b",
                text,
            )
        )
        contact_market = len(
            re.findall(
                r"Contact:|\+32\s?\d|asking\s+\d+|euro|EUR|pickup|discount|coupon|offer|sale|cash only|quick pickup|fast deal|special discount",
                text,
                flags=re.I,
            )
        )
        bad_domains = len(
            re.findall(
                r"pickup-offer-fast|trusted-sales-direct|bonus-marketplace|clearance-deals|claim-discount-now|deal-bundle-now",
                text,
                flags=re.I,
            )
        )
        html_ui = len(
            re.findall(
                r"<(?:div|span|button|input|a)\b|</(?:div|span|button|a)>",
                text,
                flags=re.I,
            )
        )
        chars = list(text)
        char_len = len(text)
        digit_ratio = sum(c.isdigit() for c in chars) / max(char_len, 1)
        upper_ratio = sum(c.isupper() for c in chars) / max(char_len, 1)
        punct_ratio = sum((not c.isalnum() and not c.isspace()) for c in chars) / max(char_len, 1)
        words = re.findall(r"[A-Za-z]+", text)
        unique_word_ratio = len(set(w.lower() for w in words)) / max(len(words), 1)

        unsafe_score = (
            10 * listing
            + 3 * urls
            + 2 * ecommerce
            + contact_market
            + 4 * bad_domains
            + html_ui
        )
        rows.append(
            {
                "char_len": char_len,
                "word_count": len(words),
                "unique_word_ratio": unique_word_ratio,
                "digit_ratio": digit_ratio,
                "upper_ratio": upper_ratio,
                "punct_ratio": punct_ratio,
                "url_count": urls,
                "listing_marker_count": listing,
                "ecommerce_token_count": ecommerce,
                "market_phrase_count": contact_market,
                "bad_domain_count": bad_domains,
                "html_ui_token_count": html_ui,
                "unsafe_score": unsafe_score,
            }
        )
    return pd.DataFrame(rows)

unsafe_features = build_unsafe_features(articles["text"])
unsafe_features.insert(0, "doc_id", articles["doc_id"])
unsafe_features.to_csv(OUTPUT_DIR / "structural_features.csv", index=False)

summary_cols = [
    "char_len",
    "word_count",
    "url_count",
    "listing_marker_count",
    "ecommerce_token_count",
    "market_phrase_count",
    "bad_domain_count",
    "unsafe_score",
]
display(unsafe_features[summary_cols].quantile([0, 0.5, 0.95, 0.98, 0.99, 1.0]))

indicator_counts = {
    "has_url": int((unsafe_features["url_count"] > 0).sum()),
    "has_listing_marker": int((unsafe_features["listing_marker_count"] > 0).sum()),
    "has_ecommerce_token": int((unsafe_features["ecommerce_token_count"] > 0).sum()),
    "has_bad_domain": int((unsafe_features["bad_domain_count"] > 0).sum()),
}
indicator_counts


## Run 3 - Analysis / rationale

The assignment asks whether groups can be expected before clustering. This is
not labeling the data; it is an exploratory check for repeated domain language.
The expected groups are later verified by unsupervised clusters and top terms.


In [ ]:
topic_probes = {
    "space_or_spaceflight": r"\b(space|nasa|orbit|moon|launch|spacecraft|lunar)\b",
    "hockey_or_sports": r"\b(hockey|nhl|team|players|playoffs|season|game)\b",
    "computer_graphics": r"\b(graphics|image|gif|jpeg|windows|file|program|software)\b",
    "autos": r"\b(car|cars|engine|dealer|ford|toyota|drive|speed)\b",
    "medical_health": r"\b(disease|doctor|patients|medical|medicine|blood|pain|treatment)\b",
    "unsafe_marketplace": r"\b(LISTING_ID_|SKU|TOKEN|COUPON|pickup|discount|offer)\b",
}

probe_rows = []
for name, pattern in topic_probes.items():
    mask = articles["text"].str.contains(pattern, case=False, regex=True, na=False)
    examples = articles.loc[mask, "doc_id"].head(5).tolist()
    probe_rows.append({"probe": name, "matching_docs": int(mask.sum()), "example_doc_ids": examples})

probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(OUTPUT_DIR / "exploratory_topic_probes.csv", index=False)
display(probe_df)


## Run 4 - Analysis / rationale

Preprocessing choices must be defensible:

- Remove stopwords because they are high-frequency function words.
- Remove very rare terms with `min_df`, because one-off tokens behave like noise.
- Remove overly common terms with `max_df`, because they do not separate topics.
- Remove signatures, e-mails, URLs, and long random IDs for clustering, because
  they are source artifacts rather than article content.
- Do not stem by default: stemming slightly reduces vocabulary but makes cluster
  explanations less readable. With TF-IDF, `min_df`, and LSA, interpretability is
  more valuable here.

The next cell demonstrates the signature-cleaning effect on an actual document.


In [ ]:
preview_ids = ["DOC_00029", "DOC_00012"]
preview_rows = []
for doc_id in preview_ids:
    raw = articles.loc[articles["doc_id"] == doc_id, "text"].iloc[0]
    cleaned = clean_text_for_clustering(raw)
    preview_rows.append(
        {
            "doc_id": doc_id,
            "raw_excerpt": re.sub(r"\s+", " ", raw[:500]),
            "cleaned_excerpt": re.sub(r"\s+", " ", cleaned[:500]),
            "raw_length": len(raw),
            "cleaned_length": len(cleaned),
        }
    )

preview_df = pd.DataFrame(preview_rows)
preview_df.to_csv(OUTPUT_DIR / "preprocessed_preview.csv", index=False)
display(preview_df)


## Run 5 - Analysis / rationale

Bag-of-words is the minimum requirement, but raw counts overweight long
documents. TF-IDF is a stronger default for this assignment because it discounts
corpus-wide terms and keeps terms that distinguish topics. The dry run below
shows how vocabulary size changes after rare/common-term filtering.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS

custom_stop = list(
    ENGLISH_STOP_WORDS.union(
        {
            "edu",
            "com",
            "article",
            "writes",
            "posting",
            "host",
            "nntp",
            "university",
            "subject",
            "lines",
            "organization",
        }
    )
)

cleaned_text = articles["text"].map(clean_text_for_clustering)
vectorizer_specs = {
    "count_bow_min_df_4": CountVectorizer(
        stop_words=custom_stop,
        min_df=4,
        max_df=0.55,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    ),
    "tfidf_min_df_4": TfidfVectorizer(
        stop_words=custom_stop,
        min_df=4,
        max_df=0.55,
        sublinear_tf=True,
        norm="l2",
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    ),
}

vectorizer_rows = []
for name, vectorizer in vectorizer_specs.items():
    matrix = vectorizer.fit_transform(cleaned_text)
    vectorizer_rows.append(
        {
            "representation": name,
            "documents": matrix.shape[0],
            "features": matrix.shape[1],
            "non_zero_entries": int(matrix.nnz),
            "density": matrix.nnz / (matrix.shape[0] * matrix.shape[1]),
        }
    )

vectorizer_df = pd.DataFrame(vectorizer_rows)
vectorizer_df.to_csv(OUTPUT_DIR / "vectorizer_dry_run.csv", index=False)
display(vectorizer_df)


## Run 5b - Analysis / rationale

The previous table can look suspicious because CountVectorizer and
TfidfVectorizer have identical `features`, `non_zero_entries`, and `density`.
That is expected when both vectorizers use the same tokenizer, stopword list,
`min_df`, and `max_df`: they keep the same vocabulary and the same document-term
positions. TF-IDF changes the values stored in those positions, not which
positions are nonzero.


In [ ]:
count_vectorizer = vectorizer_specs["count_bow_min_df_4"]
tfidf_vectorizer = vectorizer_specs["tfidf_min_df_4"]

X_count = count_vectorizer.fit_transform(cleaned_text)
X_tfidf = tfidf_vectorizer.fit_transform(cleaned_text)

same_vocabulary = count_vectorizer.vocabulary_ == tfidf_vectorizer.vocabulary_
same_nonzero_pattern = ((X_count > 0) != (X_tfidf > 0)).nnz == 0
same_values = np.allclose(X_count.data[: min(len(X_count.data), len(X_tfidf.data))], X_tfidf.data[: min(len(X_count.data), len(X_tfidf.data))])

terms = np.array(count_vectorizer.get_feature_names_out())
doc_index = 0
sample_term_indices = X_count[doc_index].nonzero()[1][:10]
value_check = pd.DataFrame(
    {
        "doc_id": articles.loc[doc_index, "doc_id"],
        "term": terms[sample_term_indices],
        "count_value": [X_count[doc_index, j] for j in sample_term_indices],
        "tfidf_value": [round(float(X_tfidf[doc_index, j]), 6) for j in sample_term_indices],
    }
)

support_check = pd.DataFrame(
    [
        {
            "same_vocabulary": same_vocabulary,
            "same_nonzero_pattern": same_nonzero_pattern,
            "same_values": same_values,
        }
    ]
)
support_check.to_csv(OUTPUT_DIR / "bow_tfidf_support_check.csv", index=False)
value_check.to_csv(OUTPUT_DIR / "bow_tfidf_value_check.csv", index=False)

display(support_check)
display(value_check)


## Run 6 - Analysis / rationale

Exploration conclusion:

1. The data has clear topical language for space, hockey, graphics, autos, and
   medical content.
2. The unsafe files are structurally distinct: marketplace/listing markers,
   repeated offer phrases, e-commerce tokens, URLs, and suspicious domains.
3. Clustering and anomaly detection should not use exactly the same features:
   clustering should emphasize topic words, while anomaly detection should
   emphasize structure and corrupted/unsafe patterns.


In [ ]:
summary_text = """Exploration summary
===================
- Validated 2164 documents and submission templates.
- Detected exactly 50 documents with the LISTING_ID_ marketplace marker.
- Saved structural features to outputs/structural_features.csv.
- Chosen clustering preprocessing: signature/url/email/code removal, English stopwords, min_df/max_df filtering, TF-IDF.
- Chosen anomaly feature family: URL/listing/e-commerce/repeated-marketplace structural indicators.
"""

(OUTPUT_DIR / "exploration_summary.txt").write_text(summary_text)
print(summary_text)
